# Lesson 21 — Practice: Logging

Work in `lessons/lesson-21-logging/` (point a terminal at that folder).

Before you start, read `lesson.md` — every construct used in these exercises is
taught there. Provided (read-only unless a spec says fix it): `noisy_shop/`
(buggy — YOU fix it), `plain_run/`, `sample.log`, `broken.json` / `good.json`,
`test_logging.py` + `test_tracked.py` (do NOT modify), `solution.py` (blank).
Installs: none — stdlib `logging` only.


## E1 — Order severities numerically

## Intent
**Compare levels like numbers: decide visibility with `>=`, never with vibes.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
Severity is arithmetic (`DEBUG 10` … `CRITICAL 50`). Write the mapping plus
one predicate, then let the Driver cell check four threshold decisions.

**File tree**

- `filters.py` (you create)

**Interface contract**

- `LEVELS` maps exactly `"DEBUG"`, `"INFO"`, `"WARNING"`, `"ERROR"`,
  `"CRITICAL"` to `10`, `20`, `30`, `40`, `50`.
- `visible(level: str, threshold: str) -> bool` returns
  `LEVELS[level] >= LEVELS[threshold]`.

**Constraints**

- **No `logging` import needed — this is pure ordering logic.**
- Driver imports `import filters as f` and checks four cases.

**Coverage**

- Levels-as-numbers → lesson.md Core Concept 1.

**Expected** (Driver cell):
```
4/4 level checks correct
```


In [1]:
import filters as f

cases = [
    (("DEBUG", "WARNING"), False),
    (("WARNING", "WARNING"), True),
    (("ERROR", "INFO"), True),
    (("CRITICAL", "CRITICAL"), True),
]
hits = sum(f.visible(level, thr) == want for (level, thr), want in cases)
print(f"{hits}/4 level checks correct")

4/4 level checks correct


## E2 — Read a log file back

## Intent
**Consume the other direction: parse a log file into per-level counts.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
`sample.log` holds lines shaped `LEVEL rest...` plus one stray line with no
level. Write the counter that reads it back.

**File tree**

- `sample.log` (provided — read only)
- `counter.py` (you create)

**Interface contract**

- `count_by_level(path: str) -> dict[str, int]` returns one entry per
  level seen, in first-seen order.
- Lines whose first token is not a known level count under `"OTHER"`.

**Constraints**

- **`encoding="utf-8"` on every open; Driver compares dicts (order-free).**

**Coverage**

- Log consumption + `encoding` discipline → lesson.md Engineering Notes
  (L9/L12 retrieval).

**Expected** (Driver cell):
```
counts correct
```


In [2]:
import counter as c

got = c.count_by_level("sample.log")
want = {"INFO": 2, "DEBUG": 2, "WARNING": 1, "ERROR": 1, "OTHER": 1}
print("counts correct" if got == want else f"MISMATCH: {got}")

counts correct


## E3 — Walk the logger tree

## Intent
**See the hierarchy: answer parent-chain questions with `is`, not guesses.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
Loggers form a dotted tree (`a.b` hangs under `a`, everything hangs under
the root). Write the one predicate that answers it.

**File tree**

- `names.py` (you create)

**Interface contract**

- `linked_child(parent_name: str, child_name: str) -> bool`:
  - creates the parent FIRST, then the child (order matters — the link
    forms at creation against existing loggers);
  - returns whether `child.parent is parent`.

**Constraints**

- **Identity (`is`), not equality — and creation order is part of the
  contract, not an accident.**
- Driver imports `import names as n` and checks three pairs.

**Coverage**

- Hierarchy + propagation prerequisites → lesson.md Core Concept 7.

**Expected** (Driver cell):
```
True False True
```


In [3]:
import names as n

print(n.linked_child("a", "a.b"), n.linked_child("c", "a.b"), n.linked_child("root", "m"))

True False True


## M1 — Route one logger to two destinations

## Intent
**Split the flow: one logger, a terse stream gate plus a verbose file gate.**

**Type:** REAL-WORLD — Medium — writes 1 file

**Task**
Build `setup()` so a named logger feeds a stream (caller-provided, for
tests) at a chosen level and a file at DEBUG — the standard dual-routing
shape from the lesson.

**File tree**

- `dual.py` (you create)

**Interface contract**

- `setup(level_name: str, path: str, stream) -> logging.Logger`:
  - returns the `"dual"` logger with level DEBUG and `propagate = False`;
  - clears pre-existing handlers first (re-runnable without duplicates);
  - attaches a `StreamHandler(stream)` at `level_name` with format
    `"%(levelname)s %(message)s"`;
  - attaches a `FileHandler(path, mode="w", encoding="utf-8")` at DEBUG
    with format `"%(levelname)s %(name)s %(message)s"`.

**Constraints**

- **Named logger `"dual"` (never root); `mode="w"` on the file.**
- Driver passes a `StringIO` stream, emits DEBUG + WARNING, and pins all
  three output lines below.

**Coverage**

- Two gates + dual routing + handler hygiene → lesson.md Core Concepts
  1/3/4.

**Expected** (Driver cell):
```
WARNING loud
DEBUG dual silent
WARNING dual loud
```


In [1]:
import io
import tempfile
from pathlib import Path

import dual as d

with tempfile.TemporaryDirectory() as td:
    p = Path(td) / "m1.log"
    buf = io.StringIO()
    logger = d.setup("WARNING", str(p), buf)
    logger.debug("silent")
    logger.warning("loud")
    print(buf.getvalue().strip())
    for line in p.read_text(encoding="utf-8").splitlines():
        print(line)
    for h in list(logger.handlers):
        h.close()
        logger.removeHandler(h)

WARNING loud
DEBUG dual silent
WARNING dual loud


## M2 — Load without crashing, log without printing

## Intent
**Fail gracefully on disk: return data-shaped defaults while the log keeps the forensics.**

**Type:** REAL-WORLD — Medium — writes 1 file

**Task**
Write the loader from the lesson's worked example: missing file, broken
JSON, and good data — three paths, zero prints, zero raises.

**File tree**

- `broken.json` / `good.json` (provided — fixtures, read only)
- `safe_load.py` (you create)

**Interface contract**

- Module logger named `"safe_load"`.
- `load_logged(path: str) -> list[dict]`:
  - missing file → logs WARNING, returns `[]`;
  - broken JSON → logs ERROR **with `exc_info=True`**, returns `[]`;
  - good file → logs INFO with the count, returns `data["items"]`.

**Constraints**

- **No `print()` anywhere in the module; `exc_info` only inside the
  `except` branch.**
- Driver attaches its own probe handler and pins returns + record
  `(level, exc-carrying?)` pairs below.

**Coverage**

- `exc_info` + data-shaped defaults + no-print rule → lesson.md Core
  Concept 5 (L9/L11 retrieval).

**Expected** (Driver cell):
```
[]
[]
[{'id': 1, 'name': 'coffee'}]
[('ERROR', True), ('WARNING', False), ('INFO', False)]
```


In [1]:
import io
import logging

import safe_load

recs = []
h = logging.StreamHandler(io.StringIO())
h.addFilter(lambda r: recs.append((r.levelname, r.exc_info is not None)) or True)
logging.getLogger("safe_load").handlers.clear()
logging.getLogger("safe_load").addHandler(h)
logging.getLogger("safe_load").setLevel(logging.DEBUG)
print(safe_load.load_logged("broken.json"))
print(safe_load.load_logged("nope.json"))
print(safe_load.load_logged("good.json"))
print(recs)

[]
[]
[{'id': 1, 'name': 'coffee'}]
[('ERROR', True), ('WARNING', False), ('INFO', False)]


## M3 — Test logging with caplog

## Intent
**Prove the log, not just the return: three `caplog` tests that pin levels and tracebacks.**

**Type:** REAL-WORLD — Medium — writes 1 test dir

**Task**
Give your M2 module the test layout it deserves: `caplog` tests that
assert what was *logged*, plus what was returned.

**File tree**

- `tests/test_load.py` (you create — exactly the three names below)

**Interface contract**

- `test_warns_on_corrupt`: corrupt fixture → returns `[]` AND a record
  with `levelno == ERROR` **and** `exc_info` set.
- `test_empty_default_on_missing`: missing path → returns `[]` AND a
  WARNING record exists.
- `test_info_on_ok`: good fixture → returns the items AND an INFO
  record exists.
- Fixtures live in `tmp_path` (written inline in each test); levels via
  `caplog.at_level(logging.DEBUG, logger="safe_load")`.

**Constraints**

- **Exactly these three test functions; assert on `caplog.records`,
  never on printed text.**
- **No `sys.path` manipulation anywhere under `tests/`.**
- Driver runs `pytest tests/test_load.py -q` and pins the count.

**Coverage**

- `caplog` as the test seam → lesson.md Practical Examples (L17 retrieval).

**Expected** (Driver cell): `3 passed`.


In [7]:
import re
import subprocess
import sys

_r = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_load.py", "-q", "-p", "no:cacheprovider"],
    capture_output=True,
    text=True,
)
_out = (_r.stdout + _r.stderr)
_m = re.search(r"(\d+ passed)", _out)
print(_m.group(1) if _m else _out[-500:])

3 passed


## H1 — Debug drill: five logging bugs in noisy_shop

## Intent
**Diagnose the observability layer itself: fix five logging bugs so each record lands exactly once, in the right place.**

**Type:** DEBUG — Hard — edits 1 tree

**Task**
`noisy_shop/` logs — noisily, twice, eagerly, traceback-less, and into
whatever folder you happen to stand in. Find and fix all five so the
suite goes green. The test file is correct — you may read it but must
not modify it.

**File tree**

- `noisy_shop/__init__.py` (provided — keep it silent)
- `noisy_shop/shop.py` (provided, buggy — YOU fix it)
- `noisy_shop/main.py` (provided, buggy — YOU fix it)
- `noisy_shop/data.csv` (provided — read only)
- `test_logging.py` (provided — do NOT modify)

**Hint (one, per your standing request — the rest is the Pitfalls list
plus the Debugging Protocol)**

- Five failures, five files-or-behaviors:
  - a config that ignores its own second call
  - a line count that doubles
  - a counter that runs when its level is off
  - a traceback that never arrives
  - a file that follows your cwd
- Map each to Common Pitfalls 1–5 yourself.

**Bug report** (fill in as you go — see the next cell).

**Coverage**

- Reconfig + duplication + laziness + `exc_info` + located paths →
  lesson.md Pitfalls 1–5 + Core Concepts 2/3/6/7; protocol →
  `notes/DEBUGGING.md`.

**Expected** (Driver cell): `5 passed` — the final state after all five fixes.


### H1 bug report — blank write-in (fill each row as you fix it, never after)

```
# | Expected vs Observed                         | Root cause | Fix
--|----------------------------------------------|------------|----
1 |                                              |            |
2 |                                              |            |
3 |                                              |            |
4 |                                              |            |
5 |                                              |            |
```


bug report:
1. a single setup_logging without using basicConfig, found multiple declaration of basicConfig instead; main.py and shop.py has its own basicConfig, which is gonna be ignored in the second call; create setup_logging in shop.py with custom configuration.
2. counter ok = 4, it should be 2; exception handling with more 'else' block to increment counter that unnecessary; delete that block.
3. debug level still ran the expensive(); no guard on isEnabledFor; add guard.
4. no traceback; no exc_info provided on except block; add exc_info=True only on except block.
5. logging setup without FileHandler; add FileHandler on setup logging

In [36]:
import re
import subprocess
import sys

_r = subprocess.run(
    [sys.executable, "-m", "pytest", "test_logging.py", "-q", "-p", "no:cacheprovider"],
    capture_output=True,
    text=True,
)
_out = (_r.stdout + _r.stderr)
_m = re.search(r"(\d+ passed)", _out)
print(_m.group(1) if _m else _out[-500:])

5 passed


## H2 — Migrate print-diagnostics to logging

## Intent
**Separate data from diagnosis: rehouse the flat patient so stdout carries exactly one line.**

**Type:** REAL-WORLD — Hard — writes 1 package + docs

**Task**
`plain_run/` prints everything (loading chatter, intermediate dicts) and
imports flat-only. Build `tracked/` where logging owns all diagnosis and
stdout carries only the data line.

**File tree**

- `plain_run/load.py` + `report.py` + `run.py` + `data.csv`
  (provided — read only)
- `tracked/__init__.py` (you create — silent)
- `tracked/load.py` (you create — guarded loader, no prints)
- `tracked/compute.py` (you create — `.get` accumulation, no prints)
- `tracked/main.py` (you create — setup + entry, prints only the
  summary line)
- `tracked/README.md` (you create — names the entry command)
- `test_tracked.py` (provided — do NOT modify)

**Interface contract**

- `load(path)` rejects non-`str`/empty paths with
  `ValueError(f"bad data path: {path!r}")` before touching disk.
- `totals(rows)` keeps `.get(team, 0.0)` accumulation (do not
  "simplify" it).
- `python -m tracked.main <data.csv>`:
  - prints exactly `Summary (2 teams, 4 rows)` to **stdout**;
  - emits INFO+ records to **stderr**, DEBUG records to a `tracked.log`
    beside the module (`mode="w"`, `encoding="utf-8"`);
  - the data line never appears on stderr.
- Wrong arity → usage on **stderr**, exit **1**.
- `README.md` names the entry command `python -m tracked.main`.

**Constraints**

- **Absolute imports, silent `__init__`, no path hacks, no `print()`
  except the single summary line — the suite checks the streams.**
- **Behavior frozen against `plain_run/data.csv`.**

**Coverage**

- Migration + stream separation + located log file → lesson.md Core
  Concepts 1/3/4/7 + Engineering Notes.

**Expected** (Driver cell): `4 passed`.


In [38]:
import re
import subprocess
import sys

_r = subprocess.run(
    [sys.executable, "-m", "pytest", "test_tracked.py", "-q", "-p", "no:cacheprovider"],
    capture_output=True,
    text=True,
)
_out = (_r.stdout + _r.stderr)
_m = re.search(r"(\d+ passed)", _out)
print(_m.group(1) if _m else _out[-500:])

4 passed


## Challenge — A ledger with a run journal

## Intent
**Ship observable software: merge with network-wins semantics while every decision lands in a leveled journal.**

**Type:** REAL-WORLD — Challenge — writes 1 project

**Task**
Build `journal/` — the neutral mini-project: feed + optional cache in,
merged JSON + CSV out, and a complete run journal beside the module.
Verbosity is a positional word, and corrupt rows are skipped loudly, not
silently.

**File tree**

- `journal/__init__.py (you create — silent)`
- `journal/main.py (you create — setup + merge + entry)`
- `journal/tests/test_journal.py (you create — exactly the 3 names below)`
- `journal/requirements.txt (you create — exactly pytest)`
- `journal/.gitignore (you create — envs, caches, data, logs)`
- `journal/README.md (you create — names the entry command)`

**Interface contract**

- `setup_verbosity(mode="normal") -> logging.Logger`:
  - configures the `"journal"` logger (never root) for `"quiet"`,
    `"normal"`, `"verbose"`;
  - the named logger itself stays at DEBUG in every mode (both gates
    set deliberately — lesson Pitfall 6);
  - clears pre-existing handlers first (re-runnable without duplicates);
  - attaches a `StreamHandler(sys.stderr)` gated by mode and a
    `FileHandler` beside the module (`mode="w"`, always DEBUG);
  - `quiet` silences the stream (CRITICAL), `normal` shows INFO+,
    `verbose` shows DEBUG+.
- `is_valid(row)`:
  - returns `isinstance(row, dict)` with `"id"`, `"price"`, `"stock"`
    all present.
- `merge(network, cached) -> (items, fresh, kept)`:
  - every network id wins;
  - cached-only items are kept;
  - output sorted by id;
  - `fresh = len(network)`.
- `python -m journal.main sync <dir> [quiet|verbose]`:
  - reads `<dir>/feed.json` + optional `<dir>/cache.json`;
  - skips invalid rows with one WARNING each, counted as `skipped`;
  - writes `journal.json` + `journal.csv` (header
    `id,name,price,stock`) into the cwd;
  - prints exactly `Ledger: N items (fresh F, cached K, skipped S)`.
- Any other argv → `usage: python -m journal.main sync <dir>
  [quiet|verbose]` on **stderr**, exit **1**, no files written.
- `journal/tests/test_journal.py` holds exactly
  `test_skip_counts_invalid`, `test_level_filter`, `test_merge_keeps_cached`
  (all green).
- `requirements.txt` is exactly `pytest` + newline.
- `.gitignore` covers envs, caches, data files, and logs.
- `README.md` names the entry command
  `python -m journal.main sync <dir>`.

**Constraints**

- **One positional verbosity word only (`quiet`/`verbose`, default
  `normal`) — no `--flags`; `newline=""` + `encoding="utf-8"` on every
  CSV handle.**
- Test 1: feed (4 valid + 1 id-bearing ghost) + cache (stale-overlap +
  offline-only) → merged ids `[1, 2, 3, 4, 5]` and
  `Ledger: 5 items (fresh 4, cached 1, skipped 1)`.
- Test 2: feed only → `Ledger: 4 items (fresh 4, cached 0, skipped 0)`.
- Test 3: bare argv → exit 1 with the usage line on stderr.
- The driver runs the entry with `cwd=tmp` + `PYTHONPATH=<lesson>`,
  your tests with `pytest journal/tests -q`, and asserts the journal
  log beside the module carries DEBUG (then removes it).

**Coverage**

- Verbosity modes + journal file + skip-loudly + project-local tests →
  lesson.md all Core Concepts; guard idioms + merge unannounced.

**Expected** (Driver cell):
```
Ledger: 5 items (fresh 4, cached 1, skipped 1)
ids: [1, 2, 3, 4, 5]
id,name,price,stock
Ledger: 4 items (fresh 4, cached 0, skipped 0)
journal log: True
usage exit: 1
3 passed
requirements: pytest
readme entry: True
```


In [ ]:
import json
import os
import re
import subprocess
import sys
import tempfile
from pathlib import Path

lesson = Path.cwd()
feed = {"items": [
    {"id": 1, "name": "coffee", "price": 2.5, "stock": 40},
    {"id": 2, "name": "tea", "price": 1.8, "stock": 25},
    {"id": 3, "name": "muffin", "price": 3.2, "stock": 0},
    {"id": 4, "name": "latte", "price": 4.1, "stock": 12},
    {"id": 9, "name": "ghost"},
]}
cache = {"items": [
    {"id": 3, "name": "muffin", "price": 0.99, "stock": 99},
    {"id": 5, "name": "donut", "price": 1.0, "stock": 3},
]}
env = {**os.environ, "PYTHONPATH": str(lesson)}
with tempfile.TemporaryDirectory() as td:
    feeddir = Path(td) / "feed"
    feeddir.mkdir()
    (feeddir / "feed.json").write_text(json.dumps(feed), encoding="utf-8")
    (feeddir / "cache.json").write_text(json.dumps(cache), encoding="utf-8")
    work = Path(td) / "work"
    work.mkdir()
    r1 = subprocess.run(
        [sys.executable, "-m", "journal.main", "sync", str(feeddir)],
        cwd=work, capture_output=True, text=True, env=env,
    )
    print(r1.stdout.strip())
    merged = json.loads((work / "journal.json").read_text(encoding="utf-8"))
    print("ids:", [it["id"] for it in merged["items"]])
    print((work / "journal.csv").read_text(encoding="utf-8").splitlines()[0])
with tempfile.TemporaryDirectory() as td:
    feeddir = Path(td) / "feed"
    feeddir.mkdir()
    (feeddir / "feed.json").write_text(
        json.dumps({"items": feed["items"][:4]}), encoding="utf-8")
    work = Path(td) / "work"
    work.mkdir()
    r2 = subprocess.run(
        [sys.executable, "-m", "journal.main", "sync", str(feeddir)],
        cwd=work, capture_output=True, text=True, env=env,
    )
    print(r2.stdout.strip())
    jl = lesson / "journal" / "journal.log"
    print("journal log:", "DEBUG" in jl.read_text(encoding="utf-8"))
    jl.unlink()
with tempfile.TemporaryDirectory() as td:
    r3 = subprocess.run(
        [sys.executable, "-m", "journal.main"],
        cwd=td, capture_output=True, text=True, env=env,
    )
    print("usage exit:", r3.returncode)
r4 = subprocess.run(
    [sys.executable, "-m", "pytest", "journal/tests", "-q", "-p", "no:cacheprovider"],
    capture_output=True, text=True, cwd=lesson,
)
_m = re.search(r"(\d+ passed)", r4.stdout + r4.stderr)
print(_m.group(1) if _m else (r4.stdout + r4.stderr)[-300:])
print("requirements:", (lesson / "journal" / "requirements.txt").read_text(encoding="utf-8").strip())
print("readme entry:", "python -m journal.main sync <dir>" in (lesson / "journal" / "README.md").read_text(encoding="utf-8"))

Ledger: 5 items (fresh 4, cached 1, skipped 1)
ids: [1, 2, 3, 4, 5]
id,name,price,stock
Ledger: 4 items (fresh 4, cached 0, skipped 0)
journal log: True
usage exit: 1
3 passed
requirements: pytest
readme entry: True
